# Finch — Phase D1/D2: Kaggle -> Clean & Split-Adjusted -> GCS (`finch-514709`)
Run in **Google Colab** (High-RAM runtime recommended).

In [ ]:
# 1. Authenticate to GCP & set project
from google.colab import auth
auth.authenticate_user()
!gcloud config set project finch-514709

In [ ]:
# 2. Clone your Finch repo & install dependencies
REPO_URL = "https://github.com/gautham9566/finch.git"
!git clone {REPO_URL} /content/finch || (cd /content/finch && git pull)
%cd /content/finch
!pip install -q -e .

In [ ]:
# 3. Upload kaggle.json (never commit this to GitHub)
from google.colab import files
import os
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/kaggle.json && chmod 600 /root/.kaggle/kaggle.json

In [ ]:
# 4. Download Kaggle 1-min dataset, validate, and push raw/clean to GCS
!python src/data/etl_kaggle_to_gcs.py
!python src/data/validate_1min.py /tmp/finch_kaggle/equities_1min.parquet

In [ ]:
# 5. Fetch splits & FII/DII, adjust 1-min prices, and upload adjusted Parquet to GCS
import duckdb
from src.data.nse_eod_fetcher import fetch_splits_yfinance, fetch_fii_dii
from src.data.corp_actions import adjust_1min_parquet

symbols = [r[0] for r in duckdb.sql("SELECT DISTINCT symbol FROM read_parquet('/tmp/finch_kaggle/equities_1min.parquet')").fetchall()]
fetch_splits_yfinance(symbols, '/tmp/finch_splits.parquet')
fetch_fii_dii('/tmp/finch_fii_dii.parquet')
adjust_1min_parquet('/tmp/finch_clean_equities/**/*.parquet', '/tmp/finch_splits.parquet', '/tmp/finch_adjusted_equities')
!gsutil -m cp -r /tmp/finch_adjusted_equities gs://finch-514709-lake/clean/equities_1min_adj/
!gsutil cp /tmp/finch_splits.parquet /tmp/finch_fii_dii.parquet gs://finch-514709-lake/raw/nse_meta/